In [24]:
# imports
from pymongo import MongoClient
from bson.objectid import ObjectId
import pandas as pd
import importlib
import requests, re, json, os, sys
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# Add parent directory to path for Jupyter notebook
sys.path.insert(0, os.path.abspath('..'))
os.environ['DEBUG'] = 'True'

from app.models.mongo_employee import Employee
from app.models.mongo_user import User
from app.models.mongo_user_permission import UserPermission
from app.core.logging import configure_logging, get_logger
from app.engine.sheets.helper import find_latest_week_ending_sheet
import app.engine.sheets.ruleset as ruleset
importlib.reload(ruleset)
RULES = ruleset.RULES
SHEET_CONFIG = ruleset.SHEET_CONFIG

configure_logging(log_level="INFO", log_format="console")
logger = get_logger(__name__)

In [3]:
# get mongodb
def get_db(uri="mongodb://localhost:27017/"):
    logger.info("Connecting to MongoDB: %s", uri)
    client = MongoClient(uri)
    logger.info("Connected to MongoDB")
    return client["zenith"]
db = get_db()

2026-07-28T16:15:37.686525Z [info     ] Connecting to MongoDB: mongodb://localhost:27017/ [__main__]
2026-07-28T16:15:37.701503Z [info     ] Connected to MongoDB           [__main__]


In [4]:
def get_excel(file_path: str, sheet_name: str = None):
    if sheet_name:
        return pd.ExcelFile(file_path, sheet_name=sheet_name)
    else:
        return pd.ExcelFile(file_path)

def df_to_excel(df: pd.DataFrame, file_path: str, sheet_name: str = None):
    if sheet_name:
        df.to_excel(file_path, index=False, sheet_name=sheet_name)
    else:
        df.to_excel(file_path, index=False)


In [5]:
BASE_PATH = "/home/phantom26/workspace/zenith/temp"
UPDATES_SHEET_PATHS = [
    f"AccessGate & API - Daily Tracker - New.xlsx",
    f"Data Team - Daily Tracker 2.xlsx",
    f"Dexia Team - Standup Status.xlsx",
    f"PMI Status Sheet (1) 1.xlsx",
    f"PMI Status Sheet 1.xlsx",
    f"Project_Phoenix_ Status Update - Platform team 2.xlsx",
    f"Project_Phoenix_ Status Update- Product Team 1.xlsx",
    f"PTR Team - Standup Status 1.xlsx"
]


In [ ]:
def get_latest_update_sheet(sheet_list: list):
    sheet_list = sheet_list[::-1]
    sheet = next(
        (s for s in sheet_list if len(re.findall(r'\d', s)) >= 3),
        None
    )
    return sheet

def get_sheet_config(sheet_name: str):
    for _, config in SHEET_CONFIG.items():
        for pattern in config["sheet_name_patterns"]:
            if re.search(pattern, sheet_name, re.IGNORECASE):
                return config
    return None

def main(excel_path: str):
    print(f"Processing {excel_path}")

    excel = get_excel(f"{BASE_PATH}/{excel_path}")
    latest_sheet_name = get_latest_update_sheet(excel.sheet_names)
    sheet_config = get_sheet_config(excel_path)
    sheet_type = sheet_config.get("sheet_type", "row_based")

    employee_index = sheet_config["column_employee_name"]["index"]
    start = sheet_config["column_week_days"]["index_start"]
    end = sheet_config["column_week_days"]["index_end"]
    header_index = sheet_config["row_column_name"]["index"]

    if sheet_type == "row_based":
        # Header exists horizontally on a row
        sheet = excel.parse(
            latest_sheet_name,
            header=header_index
        )

    elif sheet_type == "column_based":
        # Read raw because header exists vertically in a column
        sheet = excel.parse(
            latest_sheet_name,
            header=None
        )

        # Swap rows <-> columns
        sheet = sheet.T
        # Original column index is now a row index
        sheet.columns = sheet.iloc[header_index]
        sheet = sheet.iloc[header_index + 1:].reset_index(drop=True)

    else:
        raise ValueError(f"Unknown sheet_type: {sheet_type}")

    selected = sheet.iloc[
        :,
        [employee_index] + list(range(start, end + 1))
    ]

    print(selected.columns.tolist())
    print(selected.head())
    # df_to_excel(selected, f"{BASE_PATH}/{excel_path}_processed.xlsx")

for path in UPDATES_SHEET_PATHS[5:6]:
# for path in UPDATES_SHEET_PATHS:
    main(path)


Processing Project_Phoenix_ Status Update - Platform team 2.xlsx
26-07-2026
['Vlinker', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
1             Vlinker Monday Tuesday Wednesday Thursday Friday
0  Saswat Kumar Patro    NaN     NaN       NaN      NaN    NaN
1  Vaibhav Srivastava    NaN     NaN       NaN      NaN    NaN
2           R Santosh    NaN     NaN       NaN      NaN    NaN
3           R Santosh    NaN     NaN       NaN      NaN    NaN
4       Saurabh Singh    NaN     NaN       NaN      NaN    NaN
